<a href="https://colab.research.google.com/github/MarrionTechDev/Arduino-projects/blob/main/01_clean_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA = '/content/drive/MyDrive/NYPA PowerHack/data'
print(os.listdir(DATA))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['README.md', 'raw.csv']


In [8]:
import pandas as pd

RAW = DATA + '/raw.csv'
df = pd.read_csv(RAW)

print(df.shape)
display(df)
print(df.isna().sum())
print('duplicate rows:', df.duplicated().sum())

(12, 4)


,Order Date,Category,Region,Sales
0,2026-01-05,Electronics,North,120.50
1,2026-01-12,electronics,North,98.00
2,2026-01-20,Clothing,South,45.25
3,2026-02-03,Clothing,South,NaN
4,2026-02-14,Home,East,77.80
5,2026-02-14,Home,East,77.80
6,2026-03-01,Electronics,West,210.00
7,2026-03-09,HOME,North,64.10
8,2026-03-22,Clothing,East,39.99
9,2026-04-02,Electronics,South,150.75


Order Date    0
Category      0
Region        0
Sales         1
dtype: int64
duplicate rows: 1


In [9]:
# snake_case column names
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r'[^0-9a-z]+', '_', regex=True).str.strip('_'))

# trim and lowercase text columns
for c in df.columns:
    if pd.api.types.is_string_dtype(df[c]):
        df[c] = df[c].str.strip().str.lower()

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df = df.drop_duplicates()
df = df.dropna(subset=['sales'])

# checks (these stand in for the rules in DATA_CONTRACT.txt)
assert not df.duplicated().any()
assert df['sales'].notna().all()
assert df['order_date'].notna().all()
assert set(df['category']) == {'electronics', 'clothing', 'home'}

CLEAN = DATA + '/clean.csv'
df.to_csv(CLEAN, index=False)
print('wrote', CLEAN, df.shape)
display(df)

wrote /content/drive/MyDrive/NYPA PowerHack/data/clean.csv (10, 4)


,order_date,category,region,sales
0,2026-01-05,electronics,north,120.50
1,2026-01-12,electronics,north,98.00
2,2026-01-20,clothing,south,45.25
4,2026-02-14,home,east,77.80
6,2026-03-01,electronics,west,210.00
7,2026-03-09,home,north,64.10
8,2026-03-22,clothing,east,39.99
9,2026-04-02,electronics,south,150.75
10,2026-04-15,home,west,88.40
11,2026-04-28,clothing,north,52.30
